In [ ]:
import whisper
from whisper.utils import get_writer
import yt_dlp
import ffmpeg
from datetime import datetime
import tqdm
import sys
import pysrt
import re



### 1. Grab audio file from youtube video

In [ ]:
# grabbing audio file from youtube video and saving as m4a
def get_yt_audio(yt_url):
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': '%(id)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
        }]
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        error_code = ydl.download([yt_url])

    print(error_code)

# grabbing subtitles only if youtube video has them and saving as srt
def get_yt_subtitles(yt_url):
    ydl_opts = {
        'writesubtitles': True,
        # 'writeautomaticsub': True,
        'subtitlesformat': 'srt',
        'skip_download': True,
        'outtmpl': '%(id)s.%(ext)s',
        'no_cache': True,
        # 'subtitleslangs': ['en','zh-Hans'],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        error_code = ydl.download([yt_url])
    
    print(error_code)

### 2. Transcribe audio file using Whisper and add audio timestamps

In [ ]:
# Define a custom tqdm class
class _CustomProgressBar(tqdm.tqdm):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._current = self.n

    def update(self, n):
        super().update(n)
        self._current += n
        # Add your custom progress handling here (e.g., updating a GUI)
        # print(f"Progress: {self._current}/{self.total}") # Optional: for console output

def extract_yt_video_id(yt_url):
    # Extract the video ID from the YouTube URL
    if "v=" in yt_url:
        return yt_url.split("v=")[-1].split("&")[0]
    elif "youtu.be/" in yt_url:
        return yt_url.split("youtu.be/")[-1].split("?")[0]
    else:
        raise ValueError("Invalid YouTube URL")
    
def transcribe_audio(file, language):
    transcribe_module = sys.modules['whisper.transcribe']
    transcribe_module.tqdm.tqdm = _CustomProgressBar

    model = whisper.load_model("large-v2")
    decode_options = {
    "language": language,
    "verbose": False, # verbose=False is often required for the progress bar to appear/callback to work
    }
    model_opt = dict(task="transcribe", **decode_options)
    result = model.transcribe(file, **model_opt)
    writer = get_writer("srt", ".")
    writer(result, "audio.srt")
    for segment in result['segments']:
        print(f"[{segment['start']:.2f}s -> {segment['end']:.2f}s] {segment['text']}")

def process_transcription(transcription):
    blocks = transcription.split('\n\n')
    processed_lines = []
    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3:
            time_range = lines[1]
            text = lines[2]
            start_time = time_range.split(' --> ')[0]
            end_time = time_range.split(' --> ')[1]
            # Convert the time format from "00:00:00,000" to "0:00:00"
            formatted_start_time = format_time(start_time)
            formatted_end_time = format_time(end_time)
            processed_line = f"[{formatted_start_time} - {formatted_end_time}]{text}"
            processed_lines.append(processed_line)
    return '\n'.join(processed_lines)

def format_time(time):
    return datetime.strptime(time, "%H:%M:%S,%f").strftime("%H:%M:%S")

In [4]:
get_yt_audio("https://www.youtube.com/watch?v=Taw2tAMGIVM")


[youtube] Extracting URL: https://www.youtube.com/watch?v=Taw2tAMGIVM
[youtube] Taw2tAMGIVM: Downloading webpage


[youtube] Taw2tAMGIVM: Downloading android vr player API JSON
[info] Taw2tAMGIVM: Downloading 1 format(s): 140
[download] Destination: Taw2tAMGIVM.m4a
[download] 100% of  101.13MiB in 00:00:03 at 32.49MiB/s    
[FixupM4a] Correcting container of "Taw2tAMGIVM.m4a"
[ExtractAudio] Not converting audio Taw2tAMGIVM.m4a; file is already in target format m4a
0


In [12]:
# get_yt_audio("https://www.youtube.com/watch?v=Taw2tAMGIVM")
# get_yt_audio("https://www.youtube.com/watch?v=oLHfnK9IVZs")
# get_yt_subtitles("https://www.youtube.com/watch?v=oLHfnK9IVZs")
transcribe_audio("Taw2tAMGIVM.m4a", "zh")

100%|█████████▉| 652223/655223 [3:04:40<00:50, 58.86frames/s]  

[0.00s -> 3.00s] 念念我
[3.00s -> 8.00s] 詞曲 李宗盛
[30.00s -> 32.00s] 早安
[32.00s -> 34.00s] 早安
[42.00s -> 44.00s] 美貌check
[46.00s -> 48.00s] check
[48.00s -> 50.00s] 性面的这是
[50.00s -> 52.00s] 我今天幸亏没穿裙子
[52.00s -> 54.00s] 还好没穿小校服
[56.00s -> 58.00s] 欢迎你们来到东巡酒店的天空之镜观景台
[60.00s -> 64.00s] 你们身后就是威尔耸利的慕斯塔格峰
[67.00s -> 68.00s] 我们也是从这个方向来的
[68.00s -> 69.00s] 是的
[69.00s -> 70.00s] 对吧
[70.00s -> 73.00s] 然后在大家右手边是新都库什山脉
[75.00s -> 80.00s] 因此在塔线也能最直观的感受到帕米尔高原山脉的延绵交汇
[82.00s -> 84.00s] 在开始我们今天的任务之前
[84.00s -> 87.00s] 志胜你昨天是怎么回事
[88.00s -> 89.00s] 对啊
[89.00s -> 90.00s] 我也想是
[90.00s -> 93.00s] 我也没想到人在新疆后院失火了
[94.00s -> 95.00s] 说我家里乱
[95.00s -> 96.00s] 我能在实质上当朋友
[96.00s -> 97.00s] 我肯定不是杰皮
[97.00s -> 98.00s] 太吓人了
[98.00s -> 100.00s] 我都不敢往沙发上坐
[101.00s -> 102.00s] 我披个腰
[102.00s -> 103.00s] 对
[103.00s -> 104.00s] 洗脸
[104.00s -> 106.00s] 洗脸也洗手
[106.00s -> 107.00s] 洗脚
[107.00s -> 108.00s] 洗衣服
[108.00s -> 109.00s] 天天洗
[110.00s -> 112.00s] 衣服是洗过的
[112.00s -> 113.00s] 凯哥
[113.00s -> 114.00s] 看完这条热搜
[114.00s -> 115.0

In [35]:
get_yt_audio("https://www.youtube.com/watch?v=oLHfnK9IVZs")
get_yt_subtitles("https://www.youtube.com/watch?v=oLHfnK9IVZs")
transcribe_audio("oLHfnK9IVZs.m4a", "zh")

[youtube] Extracting URL: https://www.youtube.com/watch?v=oLHfnK9IVZs
[youtube] oLHfnK9IVZs: Downloading webpage


[youtube] oLHfnK9IVZs: Downloading android vr player API JSON
[info] oLHfnK9IVZs: Downloading 1 format(s): 140
[download] Destination: oLHfnK9IVZs.m4a
[download] 100% of   14.80MiB in 00:00:00 at 15.45MiB/s    
[FixupM4a] Correcting container of "oLHfnK9IVZs.m4a"
[ExtractAudio] Not converting audio oLHfnK9IVZs.m4a; file is already in target format m4a
0
[youtube] Extracting URL: https://www.youtube.com/watch?v=oLHfnK9IVZs
[youtube] oLHfnK9IVZs: Downloading webpage


[youtube] oLHfnK9IVZs: Downloading android vr player API JSON
[info] oLHfnK9IVZs: Downloading subtitles: zh-Hans
[info] oLHfnK9IVZs: Downloading 1 format(s): 248+251
[info] Writing video subtitles to: oLHfnK9IVZs.zh-Hans.srt


[download] Destination: oLHfnK9IVZs.zh-Hans.srt
[download] 100% of    8.60KiB in 00:00:00 at 71.67KiB/s
0


C:\Users\melis\AppData\Roaming\Python\Python313\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[0.00s -> 22.00s] 【 passed out for 2 days!】
[22.00s -> 27.12s] 早上好,我是家家。
[27.12s -> 34.32s] 我刚刚送了我的女儿去托尔所,
[34.32s -> 39.12s] 然后回家画了一个装。
[39.12s -> 44.00s] 现在我准备做早餐。
[44.00s -> 50.44s] 今天我打算做我最喜欢的早餐。
[50.44s -> 62.20s] 需要的食材很简单,牛油果、吐司、鸡蛋和黄油。
[62.20s -> 66.84s] 那我们先烤面包吧。
[69.84s -> 73.00s] 现在打一个鸡蛋。
[80.44s -> 84.44s] 放一点黄油。
[84.44s -> 96.44s] 黄油融快以后,把蛋倒进去。
[96.44s -> 100.44s] 然后搅拌一下。
[100.44s -> 104.44s] 接下来就很简单。
[104.44s -> 108.44s] 把牛油果抹上去。
[108.44s -> 114.44s] 牛油果和好的,可以用来水培。
[134.44s -> 144.44s] 然后再把鸡蛋放上去。
[144.44s -> 152.44s] 放一点黑胡椒。
[152.44s -> 158.44s] 放一点盐。
[159.44s -> 166.44s] 然后我个人还喜欢放一点拉茶净。
[166.44s -> 168.44s] 因为我喜欢吃辣。
[168.44s -> 172.44s] 放盘子里边。
[172.44s -> 176.44s] 现在做一杯咖啡。
[176.44s -> 182.44s] 这种咖啡糊大家都知道吧。
[183.44s -> 189.44s] 煮咖啡也很简单,只有几个步驟。
[189.44s -> 197.44s] 第一,倒水。
[197.44s -> 207.44s] 第二,放咖啡粉。
[207.44s -> 209.44s] 抖一下。
[209.44s -> 214.44s] 第三,放进去。
[214.44s -> 220.44s] 然后关上。
[220.44s -> 225.44s] 开火。
[227.44s -> 231.44s] 等个一两分钟。
[238.44s -> 244.44s] 好了。
[244.44s -> 250

### 3. Match subtitles to the video timestamps

In [ ]:
# 3. turn transcription into dict

def parse_transcription_srt(srt):
    with open(srt, "r", encoding="utf-8") as f:
        srt_content = f.read()

    content = srt_content.replace('\r\n', '\n').strip()
    blocks = re.split(r'\n\n+', content)
    transcript = []

    for block in blocks:
        lines = block.split('\n')
        if (len(lines) >= 3):
            index = lines[0].strip()

            times = lines[1].split(' --> ')

            if (len(times) == 2):
                text = " ".join(lines[2:]).strip()
                transcript.append({
                    "index": index,
                    "start_time": times[0].strip().replace(',', '.'),
                    "end_time": times[1].strip().replace(',', '.'),
                    "text": text
                })

    return transcript

In [32]:
transcript = parse_transcription_srt("audio.srt")
print(transcript)

[{'index': '1', 'start_time': '00:00:00.000', 'end_time': '00:00:03.000', 'text': '念念我'}, {'index': '2', 'start_time': '00:00:03.000', 'end_time': '00:00:08.000', 'text': '詞曲 李宗盛'}, {'index': '3', 'start_time': '00:00:30.000', 'end_time': '00:00:32.000', 'text': '早安'}, {'index': '4', 'start_time': '00:00:32.000', 'end_time': '00:00:34.000', 'text': '早安'}, {'index': '5', 'start_time': '00:00:42.000', 'end_time': '00:00:44.000', 'text': '美貌check'}, {'index': '6', 'start_time': '00:00:46.000', 'end_time': '00:00:48.000', 'text': 'check'}, {'index': '7', 'start_time': '00:00:48.000', 'end_time': '00:00:50.000', 'text': '性面的这是'}, {'index': '8', 'start_time': '00:00:50.000', 'end_time': '00:00:52.000', 'text': '我今天幸亏没穿裙子'}, {'index': '9', 'start_time': '00:00:52.000', 'end_time': '00:00:54.000', 'text': '还好没穿小校服'}, {'index': '10', 'start_time': '00:00:56.000', 'end_time': '00:00:58.000', 'text': '欢迎你们来到东巡酒店的天空之镜观景台'}, {'index': '11', 'start_time': '00:01:00.000', 'end_time': '00:01:04.000', 

In [ ]:
# 4. put all together for api

def transcribe_video(yt_url, language):
    get_yt_audio(yt_url)
    video_id = extract_yt_video_id(yt_url)
    audio_file = f"{video_id}.m4a"
    transcribe_audio(audio_file, language)
    srt_file = f"{video_id}.srt"
    transcript = parse_transcription_srt(srt_file)
    return transcript

